# 02 Error and Instability Diagnostics

연구 목적별 통합 Notebook — 승인된 매핑, 원본 스크립트 및 결과는 변경하지 않는다. 1차 Fine-Tuning까지 포함. 2차 Fine-Tuning 제외.

## 연구 구성
- [`cm.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/HGB/Normal/scripts/cm.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/HGB/Normal/logs)
- [`d2.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/HGB/Normal/scripts/d2.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/HGB/Normal/logs)
- [`extra.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/HGB/Normal/scripts/extra.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/HGB/Normal/logs)
- [`HGB_diagnosis_N5_N6.ipynb`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/HGB/Normal/scripts/colab/HGB_diagnosis_N5_N6.ipynb) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/HGB/Normal/logs)

## 사용법
원본 코드 셀은 `%%archive_source`로 등록하는 실제 원본 코드이며, 함수명 충돌을 막기 위해 현재 커널에서 임의 실행하지 않는다. 맨 아래의 실행 셀에서 원본 스크립트/Notebook을 고정 커밋의 **별도 checkout**에서 실행한다. 각 실험의 실행 인자·입력·사전 조건은 위 원본 로그/코드로 확인한다. 기존 로그·결과와 새 출력이 섞이지 않도록 독립 복사본을 사용한다. Original Test/교수님 Test를 포함한 파일은 실행 환경에 두지 않는다. 과거 스크립트가 해당 자료를 요구할 경우 실행하지 않는다. 자동 재실행 및 성능 동등성은 미검증.

원본 실행을 위한 로그 디렉터리를 별도로 지정한다.

In [ ]:
# 공통 실행 제어 — Run All은 원본 코드 등록만 하며 학습하지 않는다.
from pathlib import Path
import hashlib, json, subprocess, sys, datetime
from IPython import get_ipython
SOURCE_CELLS = {}
def archive_source(line, cell):
    SOURCE_CELLS[line.strip()] = cell
get_ipython().register_magic_function(archive_source, magic_kind="cell", magic_name="archive_source")
# 실행 시에만 설정: 기존 작업 트리가 아닌 임시 checkout (git worktree add --detach ... <ref>)
WORKTREE = None
LOG_DIR = None
RUN_PLAN = []  # 예: [("원본/스크립트.py", ["--인자", "값"])]
ALLOW_EXECUTION = False

def blob_sha(data):
    return hashlib.sha1(f"blob {len(data)}\0".encode()+data).hexdigest()

def get_verified_source(path):
    if WORKTREE is None:
        raise ValueError("해당 커밋의 별도 checkout 경로 WORKTREE를 지정하세요.")
    root=Path(WORKTREE).expanduser().resolve()
    entry=SOURCE_INDEX[path]
    head=subprocess.check_output(["git","-C",str(root),"rev-parse","HEAD"],text=True).strip()
    if head != entry["ref"]:
        raise ValueError(f"checkout ref 불일치: {head} != {entry['ref']}")
    p=root/path
    if blob_sha(p.read_bytes()) != entry["sha"]:
        raise ValueError(f"원본 Git blob 불일치: {path}")
    if path in SOURCE_CELLS:
        if p.read_text(encoding="utf-8-sig").rstrip("\n") != SOURCE_CELLS[path].rstrip("\n").lstrip("\ufeff"):
            raise ValueError(f"Notebook 보존 코드와 원본이 다릅니다: {path}")
    return root,p

def show_original(path):
    from IPython.display import display, Code
    _,p=get_verified_source(path)
    display(Code(p.read_text(encoding="utf-8"),language="python" if p.suffix==".py" else "json"))

def execute_original(path, args):
    if not ALLOW_EXECUTION:
        raise RuntimeError("명시적 실행 승인이 필요합니다.")
    root,p=get_verified_source(path)
    if LOG_DIR is None:
        raise ValueError("새 외부 LOG_DIR를 지정하세요.")
    logs=Path(LOG_DIR).expanduser().resolve();logs.mkdir(parents=True,exist_ok=True)
    if logs==root or root in logs.parents:
        raise ValueError("LOG_DIR는 checkout 밖에 두세요.")
    stamp=datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    if p.suffix==".ipynb":
        cmd=[sys.executable,"-m","jupyter","nbconvert","--to","notebook","--execute",str(p),"--output-dir",str(logs),"--output",f"{p.stem}_{stamp}.ipynb",*args]
    else:
        cmd=[sys.executable,str(p),*args]
    log=logs/f"{p.stem}_{stamp}.log"
    with log.open("x",encoding="utf-8") as f:
        f.write(json.dumps({"source":path,**SOURCE_INDEX[path],"command":cmd,"cwd":str(p.parent)},ensure_ascii=False)+"\n");f.flush()
        result=subprocess.run(cmd,cwd=p.parent,stdout=f,stderr=subprocess.STDOUT)
        f.write(f"\nreturncode={result.returncode}\n")
    print(log)
    result.check_returncode()


In [ ]:
SOURCE_INDEX = {'05_model_optimize/HGB/Normal/scripts/cm.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '027b0a65faf05ff4ad289c04d8e3fc9fc4281ed1'}, '05_model_optimize/HGB/Normal/scripts/d2.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '5f0d30a4e5fb096dc7745cd41fe0edd0ff68a06c'}, '05_model_optimize/HGB/Normal/scripts/extra.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '5993f2ef56cb09e8ad5b8fee312d8f48e2efedc8'}, '05_model_optimize/HGB/Normal/scripts/colab/HGB_diagnosis_N5_N6.ipynb': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '86951802c4f477f744ca62e74a6f641facfaa20d'}}

## cm

원본: [05_model_optimize/HGB/Normal/scripts/cm.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/HGB/Normal/scripts/cm.py)  
Git blob: `027b0a65faf05ff4ad289c04d8e3fc9fc4281ed1`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/HGB/Normal/logs)

In [ ]:
%%archive_source 05_model_optimize/HGB/Normal/scripts/cm.py
from pathlib import Path
import sys,numpy as np, matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
LAB=list('0123456789ABCDEF')
def plot(y,p,title,out):
    cm=confusion_matrix(y,p,labels=range(16)).astype(float); cm=100*cm/cm.sum(1,keepdims=True)
    fig,ax=plt.subplots(figsize=(8.6,7.6))
    im=ax.imshow(cm,cmap='viridis',vmin=0,vmax=100)
    for i in range(16):
        for j in range(16):
            v=cm[i,j]
            if v>0: ax.text(j,i,f'{v:.1f}',ha='center',va='center',fontsize=6.5,color='black' if v>60 else 'white')
    ax.set_xticks(range(16)); ax.set_xticklabels(LAB); ax.set_yticks(range(16)); ax.set_yticklabels(LAB)
    ax.set_xlabel('Predicted label'); ax.set_ylabel('True label'); ax.set_title(title,fontweight='bold',fontsize=11)
    cb=fig.colorbar(im,ax=ax,fraction=0.046,pad=0.04); cb.set_label('Percentage of true class (%)')
    plt.tight_layout(); plt.savefig(out,dpi=150); plt.close()


## d2

원본: [05_model_optimize/HGB/Normal/scripts/d2.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/HGB/Normal/scripts/d2.py)  
Git blob: `5f0d30a4e5fb096dc7745cd41fe0edd0ff68a06c`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/HGB/Normal/logs)

In [ ]:
%%archive_source 05_model_optimize/HGB/Normal/scripts/d2.py
from pathlib import Path
import sys, numpy as np
from pathlib import Path
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
REPO=Path(__file__).resolve().parents[4]; OUT=Path('out')
sys.path.insert(0,str(REPO/'05_model_optimize/MLP/Combined/scripts')); sys.path.insert(0,str(REPO/'05_model_optimize/MLP/Normal/scripts'))
from combined_mlp_writer_cv import load_official, load_writers
from exp006_size_com_comparison import process
import exp_cmb005_balanced_scale_stroke as L
off=load_official(REPO/'02_data/datasets/MNIST_EMNIST_official_split_112000.npz'); wr=load_writers(REPO/'02_data/Hand_made')
W=list(L.WRITERS); LAB='0123456789ABCDEF'; yva=off['y_val']
yhm=np.concatenate([wr[w]['y'] for w in W]); whm=np.concatenate([[w]*640 for w in W])
Xhm=np.vstack([process(wr[w]['X'],'com',20) for w in W]); Xva=process(off['X_val'],'com',20)
ph=np.load(OUT/'ph3.npy')
for tc,wc in [('D','0'),('B','8')]:
    t,w=LAB.index(tc),LAB.index(wc); bad=np.where((yhm==t)&(ph==w))[0]; good=np.where((yhm==t)&(ph==t))[0]
    n=8; fig,ax=plt.subplots(3,n+1,figsize=(2*(n+1),6.6))
    rng=np.random.default_rng(0)
    for r,(idx,ti) in enumerate([(bad,f'{tc}->{wc} wrong (n={len(bad)})'),(good,f'{tc} correct (n={len(good)})')]):
        ax[r,0].imshow(Xhm[idx].mean(0).reshape(28,28),cmap='gray'); ax[r,0].set_title('mean\n'+ti,fontsize=8)
        pick=rng.choice(idx,min(n,len(idx)),replace=False)
        for j,i in enumerate(pick): ax[r,j+1].imshow(Xhm[i].reshape(28,28),cmap='gray'); ax[r,j+1].set_title(f'writer {whm[i]}',fontsize=8)
    d=Xhm[bad].mean(0)-Xhm[good].mean(0)
    ax[2,0].imshow(d.reshape(28,28),cmap='bwr',vmin=-120,vmax=120); ax[2,0].set_title('wrong - correct\n(red: more ink in wrong)',fontsize=8)
    ax[2,1].imshow(Xva[yva==w].mean(0).reshape(28,28),cmap='gray'); ax[2,1].set_title(f'official {wc} mean',fontsize=8)
    ax[2,2].imshow(Xva[yva==t].mean(0).reshape(28,28),cmap='gray'); ax[2,2].set_title(f'official {tc} mean',fontsize=8)
    for a in ax.ravel(): a.axis('off')
    plt.tight_layout(); plt.savefig(OUT/f'D2_{tc}_to_{wc}.png',dpi=110)
    print(tc,wc,len(bad),{x:int(np.sum(whm[bad]==x)) for x in W}, 'good',{x:int(np.sum(whm[good]==x)) for x in W})


## extra

원본: [05_model_optimize/HGB/Normal/scripts/extra.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/HGB/Normal/scripts/extra.py)  
Git blob: `5993f2ef56cb09e8ad5b8fee312d8f48e2efedc8`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/HGB/Normal/logs)

In [ ]:
%%archive_source 05_model_optimize/HGB/Normal/scripts/extra.py
from pathlib import Path
import sys,time,numpy as np
stage=sys.argv[1]; sys.argv=['x','none']; exec(open(Path(__file__).resolve().parent/'run_all.py').read().split("stage=sys.argv[1]")[0])
if stage=='n1':
    Xr=off['X_train']; m=hgb(); m.fit(Xr,ytr)
    ph=m.predict(np.vstack([wr[w]['X'] for w in W])); np.save(OUT/'ph1.npy',ph); print('N1',accuracy_score(yhm,ph),m.n_iter_,flush=True)
if stage.startswith('c1_'):
    w=int(stage[3:]); o=[x for x in W if x!=w]
    X=np.vstack([Xtr]+[tw[x] for x in o]); y=np.concatenate([ytr]+[wr[x]['y'] for x in o])
    m=hgb(); m.fit(X,y); p=m.predict(tw[w]); np.save(OUT/f'c1_{w}.npy',p); print('C1',w,accuracy_score(wr[w]['y'],p),m.n_iter_,flush=True)


## HGB_diagnosis_N5_N6

원본: [05_model_optimize/HGB/Normal/scripts/colab/HGB_diagnosis_N5_N6.ipynb](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/HGB/Normal/scripts/colab/HGB_diagnosis_N5_N6.ipynb)  
Git blob: `86951802c4f477f744ca62e74a6f641facfaa20d`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/HGB/Normal/logs)

원본 Notebook의 Markdown 및 실행 코드 셀을 순서대로 보존한다. 기존 출력·이미지는 복제하지 않는다. 원본 Colab 환경/실행 순서는 원본 링크 참조.

### 원본 셀 0

# HGB 원인 분석 + N5·N6 개선 실험

**C1이 돌고 있는 노트북과 별개로, 새 Colab 창에서 동시에 돌릴 수 있게 만든 노트북입니다.** 결과는 `HGB_results_v3.csv`에 따로 저장돼서 기존 CSV와 섞이지 않아요.

| 단계 | 내용 | 시간 |
|---|---|---|
| R | N3 모델 재학습 (진단용 + 재현성 확인) | 약 35분 |
| D1 | 분기 픽셀 히트맵: 모델이 어떤 픽셀로 판단하나 | 1분 |
| D2 | 틀린 D·B vs 맞힌 D·B 이미지 비교 | 1분 |
| D3 | 작성자별 글자 크기·획 양 비교 | 1분 |
| N5 | 픽셀 + 모양 특징(투영·HOG) | 약 50분 |
| N6 | `max_features=0.5` (픽셀 과의존 줄이기) | 약 25분 |

**0번 셀 실행 → 세션 다운 메시지 → 1번 셀부터 끝까지 '모두 실행'이 아니라, 1번 셀을 클릭하고 '런타임 → 이후 셀 실행'**

### 원본 셀 1

## 0. scikit-learn 1.8.0 설치 (실행하면 세션이 재시작돼요. 정상)

In [ ]:
%%archive_source 05_model_optimize/HGB/Normal/scripts/colab/HGB_diagnosis_N5_N6.ipynb#cell2
!pip install -q scikit-learn==1.8.0
import os; os.kill(os.getpid(), 9)

### 원본 셀 3

## 1. 준비 (저장소·드라이브·데이터·N3 전처리)

In [ ]:
%%archive_source 05_model_optimize/HGB/Normal/scripts/colab/HGB_diagnosis_N5_N6.ipynb#cell4
!test -d /content/Machine_Learning || git clone --depth 1 https://github.com/ryu980920/Machine_Learning.git /content/Machine_Learning
from google.colab import drive
drive.mount('/content/drive')
import os, sys, time, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
OUT = '/content/drive/MyDrive/HGB_results/'; os.makedirs(OUT, exist_ok=True)
REPO = Path('/content/Machine_Learning')
sys.path.insert(0, str(REPO/'05_model_optimize/MLP/Combined/scripts'))
sys.path.insert(0, str(REPO/'05_model_optimize/MLP/Normal/scripts'))
from combined_mlp_writer_cv import load_official, load_writers
from exp006_size_com_comparison import process
import exp_cmb005_balanced_scale_stroke as L
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score

off = load_official(REPO/'02_data/datasets/MNIST_EMNIST_official_split_112000.npz')
wr  = load_writers(REPO/'02_data/Hand_made')
WRITERS = list(L.WRITERS); LABELS = '0123456789ABCDEF'
ytr, yva = off['y_train'], off['y_val']
yhm = np.concatenate([wr[w]['y'] for w in WRITERS])
whm = np.concatenate([[w]*len(wr[w]['y']) for w in WRITERS])

# N3와 같은 전처리
Xtr_aug, tw_com, _, _ = L.prepareraw(off, wr, 'scale30_stroke28')
Xva_com = process(off['X_val'], 'com', 20)
Xhm_com = np.vstack([tw_com[w] for w in WRITERS])
print(Xtr_aug.shape, Xva_com.shape, Xhm_com.shape)

### 원본 셀 5

## 2. 공통 함수

In [ ]:
%%archive_source 05_model_optimize/HGB/Normal/scripts/colab/HGB_diagnosis_N5_N6.ipynb#cell6
PAIRS = [('D','0'), ('0','D'), ('B','8'), ('8','B'), ('5','F'), ('7','9')]
CSV = OUT + 'HGB_results_v3.csv'
results = []

def hm_scores(pred):
    isd = yhm < 10
    r = dict(hm_acc=accuracy_score(yhm, pred), hm_f1=f1_score(yhm, pred, average='macro'),
             hm_digit=accuracy_score(yhm[isd], pred[isd]), hm_letter=accuracy_score(yhm[~isd], pred[~isd]))
    for w in WRITERS: r[f'w{w}'] = accuracy_score(yhm[whm==w], pred[whm==w])
    for a, b in PAIRS: r[f'{a}->{b}'] = int(np.sum((yhm==LABELS.index(a)) & (pred==LABELS.index(b))))
    return r

def evaluate(name, model, Xva, Xhm, extra=None):
    pv = model.predict(Xva); ph = model.predict(Xhm)
    row = dict(name=name, val_acc=accuracy_score(yva, pv), val_f1=f1_score(yva, pv, average='macro'),
               **hm_scores(ph), **(extra or {}))
    results.append(row); pd.DataFrame(results).to_csv(CSV, index=False)
    print({k:(round(v,4) if isinstance(v,float) else v) for k,v in row.items()})
    return ph

def hgb(**kw):
    p = dict(max_iter=300, early_stopping=True, validation_fraction=0.1, n_iter_no_change=10, random_state=42)
    p.update(kw); return HistGradientBoostingClassifier(**p)

def fit(m, X, y):
    t = time.time(); m.fit(X, y); return dict(fit_s=round(time.time()-t), iters=m.n_iter_)

### 원본 셀 7

## R. N3 모델 재학습
같은 데이터·설정·시드라서 어젯밤 N3(손글씨 83.44%, Validation 97.58%)와 **같은 숫자가 나와야** 해요. 같으면 재현성 확인 완료.

In [ ]:
%%archive_source 05_model_optimize/HGB/Normal/scripts/colab/HGB_diagnosis_N5_N6.ipynb#cell8
m3 = hgb(); info = fit(m3, Xtr_aug, ytr)
ph3 = evaluate('N3_재현', m3, Xva_com, Xhm_com, info)

### 원본 셀 9

## D1. 분기 픽셀 히트맵 — 모델이 어디를 보고 판단하나
트리의 모든 분기에서 쓰인 픽셀의 **gain(분기로 얻은 개선량) 합**을 28×28로 그려요. 밝을수록 그 픽셀에 많이 의존한다는 뜻이에요. 전체 + D·0·B·8 클래스 전용 트리를 따로 봐요.

In [ ]:
%%archive_source 05_model_optimize/HGB/Normal/scripts/colab/HGB_diagnosis_N5_N6.ipynb#cell10
def gain_map(model, cls=None):
    g = np.zeros(784)
    for it in model._predictors:                       # 반복마다 클래스별 트리 16개
        trees = it if cls is None else [it[cls]]
        for t in trees:
            n = t.nodes[t.nodes['is_leaf'] == 0]
            np.add.at(g, n['feature_idx'], n['gain'])
    return g.reshape(28, 28)

maps = {'All': gain_map(m3)} | {c: gain_map(m3, LABELS.index(c)) for c in 'D0B8'}
fig, ax = plt.subplots(1, 5, figsize=(20, 4.5))
for a, (k, g) in zip(ax, maps.items()):
    a.imshow(g, cmap='hot'); a.set_title(f'{k} trees'); a.axis('off')
plt.suptitle('N3 HGB split-gain heatmap'); plt.tight_layout()
plt.savefig(OUT+'D1_gain_heatmap.png', dpi=130); plt.show()

# 상위 몇 개 픽셀에 의존이 몰렸는지 수치로
for k, g in maps.items():
    s = np.sort(g.ravel())[::-1]; top = s[:20].sum()/s.sum()
    used = (g > 0).sum()
    print(f'{k}: 사용 픽셀 {used}/784, 상위 20픽셀이 전체 gain의 {top:.1%}')

### 원본 셀 11

## D2. 틀린 D·B vs 맞힌 D·B 이미지 + 평균 이미지

In [ ]:
%%archive_source 05_model_optimize/HGB/Normal/scripts/colab/HGB_diagnosis_N5_N6.ipynb#cell12
def show_cases(true_c, wrong_c, n=8):
    t, w = LABELS.index(true_c), LABELS.index(wrong_c)
    bad  = np.where((yhm==t) & (ph3==w))[0]; good = np.where((yhm==t) & (ph3==t))[0]
    fig, ax = plt.subplots(3, n+1, figsize=(2*(n+1), 6.5))
    rows = [(bad, f'{true_c}→{wrong_c} (wrong)'), (good, f'{true_c} correct')]
    for r, (idx, title) in enumerate(rows):
        ax[r,0].imshow(Xhm_com[idx].mean(0).reshape(28,28), cmap='gray'); ax[r,0].set_title('mean\n'+title, fontsize=8)
        for j in range(n):
            if j < len(idx):
                ax[r,j+1].imshow(Xhm_com[idx[j]].reshape(28,28), cmap='gray'); ax[r,j+1].set_title(f'w{whm[idx[j]]}', fontsize=8)
    # 3행: 틀린 평균 - 맞힌 평균 (빨강=틀린 쪽에 더 많은 획)
    diff = Xhm_com[bad].mean(0) - Xhm_com[good].mean(0)
    ax[2,0].imshow(diff.reshape(28,28), cmap='bwr', vmin=-120, vmax=120); ax[2,0].set_title('wrong − correct', fontsize=8)
    ax[2,1].imshow(Xva_com[yva==w].mean(0).reshape(28,28), cmap='gray'); ax[2,1].set_title(f'official {wrong_c} mean', fontsize=8)
    ax[2,2].imshow(Xva_com[yva==t].mean(0).reshape(28,28), cmap='gray'); ax[2,2].set_title(f'official {true_c} mean', fontsize=8)
    for a in ax.ravel(): a.axis('off')
    plt.tight_layout(); plt.savefig(OUT+f'D2_{true_c}_to_{wrong_c}.png', dpi=130); plt.show()
    print(f'{true_c}→{wrong_c}: {len(bad)}건 / 작성자별', {w_: int(np.sum(whm[bad]==w_)) for w_ in WRITERS})

show_cases('D', '0'); show_cases('B', '8')

### 원본 셀 13

## D3. 작성자별 글자 특성 (크기·획 양·가로세로 비율) vs 공식 자료

In [ ]:
%%archive_source 05_model_optimize/HGB/Normal/scripts/colab/HGB_diagnosis_N5_N6.ipynb#cell14
def geom(X, th=32):
    X = X.reshape(-1, 28, 28); out = []
    for im in X:
        ys, xs = np.where(im >= th)
        if len(ys) == 0: out.append((0,0,0,0)); continue
        h, w = np.ptp(ys)+1, np.ptp(xs)+1
        out.append((max(h, w), w/h, len(ys), len(ys)/(h*w)))   # 긴 변, 가로/세로, 획 픽셀 수, 채움 비율
    return pd.DataFrame(out, columns=['long_side', 'w_over_h', 'ink_px', 'fill_ratio'])

rows = []
for name, X, y in [('official_val', off['X_val'], yva)] + [(f'writer{w}', wr[w]['X'], wr[w]['y']) for w in WRITERS]:
    g = geom(X); g['is_letter'] = y >= 10
    for part, sub in [('digit', g[~g.is_letter]), ('letter', g[g.is_letter])]:
        rows.append(dict(source=name, part=part, **sub.drop(columns='is_letter').mean().round(2).to_dict()))
geo = pd.DataFrame(rows); geo.to_csv(OUT+'D3_writer_geometry.csv', index=False); geo

### 원본 셀 15

## N5. 픽셀 + 모양 특징 (가로·세로 투영 56개 + HOG 324개)
**가설:** 트리는 픽셀 하나로 분기해서 획이 1픽셀만 밀려도 판단이 바뀐다. 획 방향(HOG)과 줄별 획 양(투영)은 작은 이동에 덜 흔들리므로, D의 왼쪽 직선·B의 왼쪽 세로선을 더 안정적으로 잡을 것이다.

In [ ]:
%%archive_source 05_model_optimize/HGB/Normal/scripts/colab/HGB_diagnosis_N5_N6.ipynb#cell16
from skimage.feature import hog

def shape_feats(X):
    X = X.reshape(-1, 28, 28).astype(np.float32)
    proj = np.hstack([X.sum(1), X.sum(2)]) / 255.0                       # 세로 투영 28 + 가로 투영 28
    H = np.stack([hog(im, orientations=9, pixels_per_cell=(7,7), cells_per_block=(2,2)) for im in X])
    return np.hstack([proj, H]).astype(np.float32)

t = time.time()
F_tr, F_va, F_hm = shape_feats(Xtr_aug), shape_feats(Xva_com), shape_feats(Xhm_com)
print('특징 계산', round(time.time()-t), '초, 특징 수', F_tr.shape[1])

m5 = hgb(); info = fit(m5, np.hstack([Xtr_aug, F_tr]), ytr)
ph5 = evaluate('N5_pixel+proj+HOG', m5, np.hstack([Xva_com, F_va]), np.hstack([Xhm_com, F_hm]),
               {**info, 'n_features': 784 + F_tr.shape[1]})

### 원본 셀 17

## N6. `max_features=0.5` — 분기마다 픽셀 절반만 후보로
**가설:** D1에서 소수 픽셀에 gain이 몰려 있다면, 분기 후보를 무작위로 절반만 보게 해서 여러 픽셀에 판단을 분산시키면 손글씨 변형에 강해질 것이다.

In [ ]:
%%archive_source 05_model_optimize/HGB/Normal/scripts/colab/HGB_diagnosis_N5_N6.ipynb#cell18
m6 = hgb(max_features=0.5); info = fit(m6, Xtr_aug, ytr)
ph6 = evaluate('N6_max_features=0.5', m6, Xva_com, Xhm_com, {**info, 'max_features': 0.5})

g6 = gain_map(m6); g3 = maps['All']
for k, g in [('N3', g3), ('N6', g6)]:
    s = np.sort(g.ravel())[::-1]; print(f'{k}: 상위 20픽셀 gain 비중 {s[:20].sum()/s.sum():.1%}')

### 원본 셀 19

## 결과 비교 + 혼동행렬

In [ ]:
%%archive_source 05_model_optimize/HGB/Normal/scripts/colab/HGB_diagnosis_N5_N6.ipynb#cell20
from sklearn.metrics import ConfusionMatrixDisplay
fig, ax = plt.subplots(1, 3, figsize=(24, 8))
for a, (p, t) in zip(ax, [(ph3,'N3 (baseline)'), (ph5,'N5 pixel+shape'), (ph6,'N6 max_features=0.5')]):
    ConfusionMatrixDisplay.from_predictions(yhm, p, labels=range(16), display_labels=list(LABELS),
                                            normalize='true', values_format='.2f', ax=a, colorbar=False)
    a.set_title(t)
plt.tight_layout(); plt.savefig(OUT+'N3_N5_N6_confusion.png', dpi=120); plt.show()
pd.DataFrame(results)

## 실험 실행 및 결과 확인

`RUN_PLAN`에 실제 코드의 CLI 인자와 실행 순서를 넣고, `ALLOW_EXECUTION=True`를 설정한 경우에만 실행된다. 이전 결과를 다시 계산한 것과 새 학습 결과를 구별한다.

In [ ]:
if RUN_PLAN:
    for source_path, arguments in RUN_PLAN:
        execute_original(source_path, arguments)
else:
    print("원본 코드 등록 완료. 학습/평가는 실행하지 않았습니다.")